# Function 5: Black-box optimisation

This notebook documents 13 rounds of model fitting, candidate selection and evaluated outputs for Function 5. Alternative approaches are retained where they informed the weekly decisions.

**Main result:** The upper-corner query reached 8,662.4825 in Week 7; later tests investigated input symmetry.

**Usage:** Run the setup and then the weekly sections in order. The initial datasets are in this folder; the full query history is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv). Some early searches generate large candidate grids and may require substantial memory.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from itertools import product
from scipy.optimize import NonlinearConstraint, differential_evolution, least_squares
from scipy.spatial import cKDTree
from scipy.stats import qmc, truncnorm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF


def resolve_initial_path(filename: str) -> Path:
    """Find the initial dataset for Function 5."""
    for folder in (Path.cwd(), Path.cwd() / "function_5"):
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"{filename} not found. Place it in the function_5/ folder."
    )

## Reusable routines

The helper functions below handle standard GP fitting, UCB scoring and recording evaluated queries. The model settings and experiments remain visible within each week. Global/local candidate generation is shared across the early rounds.

In [ ]:
def fit_gp_model(X, y, **settings):
    """Fit a GP using the explicit kernel and settings chosen for a given week."""
    gp = GaussianProcessRegressor(**settings)
    gp.fit(X, y)
    return gp


def ucb_score(mean, std, beta):
    """Upper Confidence Bound: predictive mean plus weighted uncertainty."""
    return mean + beta * std


def add_evaluation(X, y, x_observed, y_observed):
    """Add one completed black-box evaluation to the training history."""
    x_row = np.asarray(x_observed, dtype=float).reshape(1, -1)
    y_value = np.asarray(y_observed, dtype=float).reshape(1)
    if x_row.shape[1] != X.shape[1]:
        raise ValueError("The evaluated input has the wrong number of dimensions.")
    return np.vstack([X, x_row]), np.concatenate([y, y_value])

def sample_hybrid_candidates(best_x, n_global, n_local, seed, *,
                             local_scale=0.08, local_method="truncated"):
    """Combine uniform global samples with local samples near the current best.

    The local method is either "truncated" (truncated normal, inside [0, 1])
    or "clipped" (normal samples clipped to [0, 1]). Random number generation
    follows the same sequence for a given seed as the weekly experiments.
    """
    best_x = np.asarray(best_x, dtype=float).reshape(-1)
    n_features = best_x.size
    rng = np.random.default_rng(seed)

    X_global = rng.uniform(0, 1, size=(n_global, n_features))
    if local_method == "clipped":
        X_local = rng.normal(
            loc=best_x, scale=local_scale, size=(n_local, n_features)
        )
        X_local = np.clip(X_local, 0, 1)
    elif local_method == "truncated":
        X_local = np.empty((n_local, n_features))
        for j in range(n_features):
            a = (0 - best_x[j]) / local_scale
            b = (1 - best_x[j]) / local_scale
            X_local[:, j] = truncnorm.rvs(
                a, b, loc=best_x[j], scale=local_scale,
                size=n_local, random_state=rng
            )
    else:
        raise ValueError("local_method must be 'clipped' or 'truncated'.")

    return np.round(np.vstack([X_global, X_local]), 6)


## Starting data

In [2]:
X = np.load(resolve_initial_path("initial_inputs.npy"))
y = np.load(resolve_initial_path("initial_outputs.npy"))

## Week 1

In [6]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

# UCB exploration parameter
beta = 1.96
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds="fixed"
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True
)
# Generate candidate inputs
# Define the search range for each input variable

x1_values = np.linspace(0, 1, 70)
x2_values = np.linspace(0, 1, 70)
x3_values = np.linspace(0, 1, 70)
x4_values = np.linspace(0, 1, 70)

# Generate every combination of x1 and x2
X1_grid, X2_grid, X3_grid, X4_grid = np.meshgrid(
    x1_values,
    x2_values,
    x3_values,
    x4_values
)

X_grid = np.column_stack([
    X1_grid.ravel(),
    X2_grid.ravel(),
    X3_grid.ravel(),
    X4_grid.ravel()
])
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_grid,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)
# Select the next query
best_index = np.argmax(UCB)

next_X = X_grid[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.20289855 0.85507246 0.82608696 0.86956522]

Predicted mean:
965.3540792588839

Predicted standard deviation:
130.40260169054696

UCB:
1220.943178572356


### Evaluated query

**Input:** `(0.202899, 0.855072, 0.826087, 0.869565)`  
**Output:** `865.9944200425758`

In [ ]:
X_week1 = np.array([0.202899, 0.855072, 0.826087, 0.869565])
y_week1 = np.array([865.9944200425758])
X_new, y_new = add_evaluation(X, y, X_week1, y_week1)

## Week 2

In [10]:
X = X_new
y = y_new

### Increase β to favour exploration

In [11]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 3

### Allow the kernel length scale to vary

In [12]:
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.0933)


### Use random candidates instead of a grid

In [13]:
# Generate candidate inputs
n_candidates = 1000000
n_features = 4

np.random.seed(42)

X_candidates = np.random.uniform(
    low=0,
    high=1,
    size=(n_candidates, n_features)
)

X_candidates = np.round(X_candidates, 6)

In [14]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [15]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.227357 0.79672  0.874691 0.930961]

Predicted mean:
864.154931478557

Predicted standard deviation:
189.73492593606176

UCB:
1433.3597092867424


### Evaluated query

**Input:** `(0.227357, 0.796720, 0.874691, 0.930961)`  
**Output:** `1143.9206750696114`

In [ ]:
X_week2 = np.array([0.227357, 0.79672 , 0.874691, 0.930961])
y_week2 = np.array([1143.9206750696114])
X_new, y_new = add_evaluation(X, y, X_week2, y_week2)

## Week 3

In [19]:
X = X_new
y = y_new

### Option 1: Isotropic GP with global/local sampling · selected

In [20]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 2.5

In [21]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.225)


In [22]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="truncated",
)

In [23]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [24]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.404291 0.811542 0.997777 0.936252]

Predicted mean:
1202.5507177765282

Predicted standard deviation:
213.18226118672533

UCB:
1735.5063707433414


### Option 2: Anisotropic GP with global/local sampling

In [25]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1]

beta = 2.5

In [26]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[2, 2, 0.206, 0.294])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [27]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="truncated",
)

In [28]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [29]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.849739 0.999728 0.997544 0.997856]

Predicted mean:
1357.924881087487

Predicted standard deviation:
142.72998995186194

UCB:
1714.7498559671417


### Evaluated query

**Input:** `(0.404291, 0.811542, 0.997777, 0.936252)`  
**Output:** `2148.6500967829866`

In [ ]:
X_week3 = np.array([0.404291, 0.811542, 0.997777, 0.936252])
y_week3 = np.array([2148.6500967829866])
X_new, y_new = add_evaluation(X, y, X_week3, y_week3)

## Week 4

In [33]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 0.5 · selected

In [34]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 0.5

In [35]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.381)


In [36]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [37]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [38]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.56744  0.818244 0.999466 0.989229]

Predicted mean:
2374.736701770499

Predicted standard deviation:
121.1999543551503

UCB:
2435.336678948074


### Option 2: Anisotropic GP with β = 0.5, change upper bound length scale for x2 · alternative

In [39]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1]

beta = 0.5

In [40]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=[
        (0.01, 2.0),   # x1
        (0.01, 10.0),  # x2
        (0.01, 2.0),   # x3
        (0.01, 2.0)    # x4
    ]
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.189, 10, 0.411, 0.757])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [41]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [42]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [43]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.393282 0.772958 0.998975 0.999815]

Predicted mean:
2202.454634113646

Predicted standard deviation:
28.0722401350529

UCB:
2216.4907541811726


### Evaluated query

**Input:** `(0.567440, 0.818244, 0.999466, 0.989229)`  
**Output:** `2909.8953273073485`

In [ ]:
X_week4 = np.array([0.56744, 0.818244, 0.999466, 0.989229])
y_week4 = np.array([2909.8953273073485])
X_new, y_new = add_evaluation(X, y, X_week4, y_week4)

## Week 5

In [47]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 0.5 · selected

In [48]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 0.5

In [49]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.443)


In [50]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=47,
    local_scale=0.08,
    local_method="truncated",
)

In [51]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [52]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.804168 0.91374  0.988463 0.99964 ]

Predicted mean:
3407.561683252681

Predicted standard deviation:
199.75738149962484

UCB:
3507.4403740024936


### Evaluated query

**Input:** `(0.804168, 0.913740, 0.988463, 0.999640)`  
**Output:** `4727.1230277053555`

In [ ]:
X_week5 = np.array([0.804168, 0.91374 , 0.988463, 0.99964])
y_week5 = np.array([4727.1230277053555])
X_new, y_new = add_evaluation(X, y, X_week5, y_week5)

## Week 6

In [56]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 0.5 · selected

In [57]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 0.5

In [58]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.471)


In [59]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=48,
    local_scale=0.08,
    local_method="truncated",
)

In [60]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [61]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.993657 0.99954  0.99845  0.980099]

Predicted mean:
5824.669228948964

Predicted standard deviation:
196.56269135478706

UCB:
5922.950574626358


### Evaluated query

**Input:** `(0.993657, 0.999540, 0.998450, 0.980099)`  
**Output:** `8135.758394027195`

In [ ]:
X_week6 = np.array([0.993657, 0.99954 , 0.99845 , 0.980099])
y_week6 = np.array([8135.758394027195])
X_new, y_new = add_evaluation(X, y, X_week6, y_week6)

## Week 7

In [65]:
# All observations through Week 6
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).reshape(-1)

assert X.shape == (26, 4)
assert y.shape == (26,)
assert np.all(y >= 0)

# Square-root transform
y_transformed = np.sqrt(y)

# Tuned GP: amplitude * isotropic RBF
kernel = ConstantKernel(
    constant_value=1.0,
    constant_value_bounds=(0.03, 1000.0)
) * RBF(
    length_scale=0.45,
    length_scale_bounds=(0.02, 10.0)
)

model = fit_gp_model(
    X, y_transformed,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Optimised kernel:", model.kernel_)

# GP-UCB in square-root space
beta = 0.5

def negative_ucb(points):
    # Differential evolution passes (features, candidates)
    candidates = points.T

    mu, std = model.predict(
        candidates,
        return_std=True
    )

    return -(ucb_score(mu, std, beta))

result = differential_evolution(
    negative_ucb,
    bounds=[(0, 1)] * 4,
    seed=48,
    popsize=12,
    maxiter=120,
    tol=1e-6,
    vectorized=True,
    updating="deferred",
    polish=False
)

# Also evaluate the exact upper-bound corner
candidates = np.vstack([
    result.x,
    np.ones(4)
])

mu, std = model.predict(
    candidates,
    return_std=True
)

ucb = ucb_score(mu, std, beta)

# Exclude exact or near-duplicate evaluations
distances = cKDTree(X).query(candidates)[0]
ucb[distances < 0.01] = -np.inf

best_index = np.argmax(ucb)
next_X = candidates[best_index]

# Predict on original scale
mu_next = mu[best_index]
std_next = std[best_index]

predicted_y = mu_next**2 + std_next**2

# Standard deviation of squared normal variable
predicted_y_std = np.sqrt(
    4 * mu_next**2 * std_next**2
    + 2 * std_next**4
)

print("Week 7 recommended input:", np.round(next_X, 6))
print("Predicted output:", predicted_y)
print("Approx. predicted std:", predicted_y_std)
print("Nearest observation distance:", distances[best_index])

Optimised kernel: 1.8**2 * RBF(length_scale=0.696)
Week 7 recommended input: [1. 1. 1. 1.]
Predicted output: 8355.732670723019
Approx. predicted std: 133.31943698867806
Nearest observation distance: 0.020949881861242033


### Evaluated query

**Input:** `(1.000000, 1.000000, 1.000000, 1.000000)`  
**Output:** `8662.4825`

In [ ]:
X_week7 = np.array([1., 1., 1., 1.])
y_week7 = np.array([8662.4825])
X_new, y_new = add_evaluation(X, y, X_week7, y_week7)

## Week 8

In [69]:
# All observations through Week 7
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (27, 4)
assert y.shape == (27,)
assert np.all(y >= 0)

# Square-root transformation
y_sqrt = np.sqrt(y)

# Three kernel configurations
kernels = [
    # Model 1: isotropic Matérn 5/2
    ConstantKernel(1.0, (0.03, 1000.0))
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 10.0),
        nu=2.5
    ),

    # Model 2: isotropic RBF
    ConstantKernel(1.0, (0.03, 1000.0))
    * RBF(
        length_scale=0.5,
        length_scale_bounds=(0.02, 10.0)
    ),

    # Model 3: anisotropic RBF
    ConstantKernel(1.0, (0.03, 1000.0))
    * RBF(
        length_scale=np.full(4, 0.5),
        length_scale_bounds=(0.02, 10.0)
    )
]

weights = np.array([0.5, 0.3, 0.2])

models = []

for kernel in kernels:
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y_sqrt)
    models.append(gp)

    print("Fitted kernel:", gp.kernel_)

# Ensemble prediction in square-root space
def ensemble_predict(points):
    predictions = [
        model.predict(points, return_std=True)
        for model in models
    ]

    means = np.array([p[0] for p in predictions])
    variances = np.array([p[1]**2 for p in predictions])

    mu = np.average(
        means, axis=0, weights=weights
    )

    variance = np.average(
        variances + (means - mu)**2,
        axis=0,
        weights=weights
    )

    return mu, np.sqrt(variance)

# Acquisition settings
beta = 0.5
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(points):
    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = ensemble_predict(candidates)
    distances = tree.query(candidates)[0]

    # Strongly discourage near-duplicate queries
    penalty = 1e6 * np.maximum(
        0, min_distance - distances
    )

    scores = -(ucb_score(mu, std, beta)) + penalty

    return scores if points.ndim == 2 else scores[0]

# Optimise acquisition from multiple seeds
solutions = []

for seed in [42, 48, 59]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 4,
        seed=seed,
        popsize=15,
        maxiter=220,
        tol=1e-7,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    solutions.append(result)

best_result = min(solutions, key=lambda r: r.fun)
next_X = np.round(best_result.x, 6)

mu, std = ensemble_predict(
    next_X.reshape(1, -1)
)

# Transform predictions back to original scale
predicted_y = mu[0]**2 + std[0]**2

# Approximate standard deviation on original scale
predicted_y_std = np.sqrt(
    4 * mu[0]**2 * std[0]**2
    + 2 * std[0]**4
)

print("Week 8 query:", next_X)
print("Predicted output:", predicted_y)
print("Approx. predicted std:", predicted_y_std)
print("UCB (square-root space):",
      mu[0] + beta * std[0])
print("Nearest distance:",
      tree.query(next_X)[0])

Fitted kernel: 3.35**2 * Matern(length_scale=1.94, nu=2.5)
Fitted kernel: 1.9**2 * RBF(length_scale=0.754)
Fitted kernel: 1.66**2 * RBF(length_scale=[0.529, 1.88, 0.635, 0.744])
Week 8 query: [1.   0.95 1.   1.  ]
Predicted output: 8111.157560552631
Approx. predicted std: 336.21864867541456
UCB (square-root space): 90.97601401224135
Nearest distance: 0.050000000000000044


### Evaluated query

**Input:** `(1.000000, 0.950000, 1.000000, 1.000000)`  
**Output:** `7786.3713496896`

In [ ]:
X_week8 = np.array([1.  , 0.95, 1.  , 1.])
y_week8 = np.array([7786.3713496896])
X_new, y_new = add_evaluation(X, y, X_week8, y_week8)

## Week 9

In [73]:
# All observations through Week 8
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (28, 4)
assert y.shape == (28,)
assert np.all(y >= 0)

# Square-root transformation
y_sqrt = np.sqrt(y)

# Selected GP model
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 1000.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 10.0),
        nu=2.5
    )
)

model = fit_gp_model(
    X, y_sqrt,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)
print("Optimised kernel:", model.kernel_)

# Acquisition settings
beta = 0.5
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(points):
    points = np.asarray(points)

    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = model.predict(
        candidates,
        return_std=True
    )

    distances = tree.query(candidates)[0]

    penalty = 10000 * np.maximum(
        0, min_distance - distances
    )

    scores = -(ucb_score(mu, std, beta)) + penalty

    return scores if points.ndim == 2 else scores[0]

# Optimise acquisition function
solutions = []

for seed in [42, 59]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 4,
        seed=seed,
        popsize=12,
        maxiter=160,
        tol=1e-7,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    solutions.append(result)

best_result = min(
    solutions,
    key=lambda r: r.fun
)

next_X = np.round(best_result.x, 6)

# Predict in square-root space
mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

# Back-transform predictive moments
predicted_y = mu[0]**2 + std[0]**2

predicted_y_std = np.sqrt(
    4 * mu[0]**2 * std[0]**2
    + 2 * std[0]**4
)

print("Week 9 query:", next_X)
print("Predicted output:", predicted_y)
print("Approx. predicted std:", predicted_y_std)
print("UCB (square-root space):",
      mu[0] + beta * std[0])
print("Nearest distance:", tree.query(next_X)[0])
print("Current best:", y.max())

Optimised kernel: 3.28**2 * Matern(length_scale=2.03, nu=2.5)
Week 9 query: [1.   1.   0.95 1.  ]
Predicted output: 7926.102862532158
Approx. predicted std: 229.5641346382796
UCB (square-root space): 89.66399564577198
Nearest distance: 0.050000000000000044
Current best: 8662.4825


### Evaluated query

**Input:** `(1.000000, 1.000000, 0.950000, 1.000000)`  
**Output:** `7786.3713496896`

In [ ]:
X_week9 = np.array([1.  , 1.  , 0.95, 1.  ])
y_week9 = np.array([7786.3713496896])
X_new, y_new = add_evaluation(X, y, X_week9, y_week9)

## Week 10

In [77]:
# All observations through Week 9
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (29, 4)
assert y.shape == (29,)
assert np.all(y >= 0)

# Check the unusual repeated observation
if np.isclose(y[-1], y[-2], rtol=0, atol=1e-12):
    print(
        "WARNING: Weeks 8 and 9 have identical "
        "recorded outputs. Verify Week 9."
    )

# Square-root transformation
y_sqrt = np.sqrt(y)

# Preferred GP
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 1000.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 10.0),
        nu=2.5
    )
)

model = fit_gp_model(
    X, y_sqrt,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)
print("Fitted kernel:", model.kernel_)

# Acquisition settings
beta = 0.5
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(x):
    x = np.asarray(x).reshape(1, -1)

    mean, std = model.predict(
        x,
        return_std=True
    )

    return -float(mean[0] + beta * std[0])

# A small margin protects the constraint
# when the candidate is rounded for submission.
distance_constraint = NonlinearConstraint(
    lambda x: tree.query(x)[0],
    min_distance + 1.5e-5,
    np.inf
)

# Global acquisition optimisation
solutions = []

for seed in [42, 59, 111]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 4,
        constraints=distance_constraint,
        seed=seed,
        popsize=12,
        maxiter=140,
        tol=1e-7,
        polish=True
    )

    if tree.query(result.x)[0] >= (
        min_distance + 1e-5
    ):
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible candidate found."
    )

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)

# Predict in square-root space
mean, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

# Back-transform the predictive moments
predicted_y = mean[0]**2 + std[0]**2

predicted_std = np.sqrt(
    4 * mean[0]**2 * std[0]**2
    + 2 * std[0]**4
)

print("Week 10 query:", next_X)
print("Predicted output:", predicted_y)
print("Approx. std:", predicted_std)
print("GP-UCB:", mean[0] + beta * std[0])
print(
    "Nearest distance:",
    tree.query(next_X)[0]
)
print("Current best:", y.max())

Fitted kernel: 3.37**2 * Matern(length_scale=2.16, nu=2.5)


/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_trustregion_constr/equality_constrained_sqp.py:217: UserWarning: Singular Jacobian matrix. Using SVD decomposition to perform the factorizations.
  Z, LS, Y = projections(A, factorization_method)
/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_trustregion_constr/equality_constrained_sqp.py:217: UserWarning: Singular Jacobian matrix. Using SVD decomposition to perform the factorizations.
  Z, LS, Y = projections(A, factorization_method)
/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_trustregion_constr/equality_constrained_sqp.py:217: UserWarning: Singular Jacobian matrix. Using SVD decomposition to perform the factorizations.
  Z, LS, Y = projections(A, factorization_method)
/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_trustregion_constr/equality_constrained_sqp.py:217: UserWarning: Singular Jacobian matrix. Using SVD decomposition to perform the factorizations.
  Z, LS, Y = projections(A

Week 10 query: [0.9478 1.     1.     1.    ]
Predicted output: 7791.373265354777
Approx. std: 41.21024659724328
GP-UCB: 88.38516551512718
Nearest distance: 0.05001528116486002
Current best: 8662.4825


**Week 10 selection:** I submitted `x₁ = 0.950000` to compare the effect of reducing each coordinate independently from the upper corner.

### Evaluated query

**Input:** `(0.950000, 1.000000, 1.000000, 1.000000)`  
**Output:** `7786.3713496896`

In [ ]:
X_week10 = np.array([0.95, 1.  , 1.  , 1.])
y_week10 = np.array([7786.3713496896])
X_new, y_new = add_evaluation(X, y, X_week10, y_week10)

## Week 11

In [80]:
# Observations through Week 10
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (30, 4)
assert y.shape == (30,)
assert np.all(y >= 0)

best_idx = np.argmax(y)

print("Best observed input:", X[best_idx])
print("Best observed output:", y[best_idx])
# Square-root transform
y_sqrt = np.sqrt(y)
# Fit two GP models
kernels = {
    "Matern 5/2": (
        ConstantKernel(1.0, (0.01, 1000.0))
        * Matern(
            length_scale=0.5,
            length_scale_bounds=(0.02, 10.0),
            nu=2.5
        )
    ),
    "RBF": (
        ConstantKernel(1.0, (0.01, 1000.0))
        * RBF(
            length_scale=0.5,
            length_scale_bounds=(0.02, 10.0)
        )
    )
}

models = {}

for name, kernel in kernels.items():

    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-7,
        normalize_y=True,
        n_restarts_optimizer=8,
        random_state=42
    )

    model.fit(X, y_sqrt)
    models[name] = model

    print(name, ":", model.kernel_)
# Compare four symmetric probes
candidates = np.ones((4, 4))

for i in range(4):
    candidates[i, i] = 0.95

print("\nSymmetry probe predictions:")

for name, model in models.items():

    mu, std = model.predict(
        candidates,
        return_std=True
    )

    # Back-transform the first two moments
    predicted_y = mu**2 + std**2

    print("\n", name)

    for point, prediction in zip(
        candidates, predicted_y
    ):
        print(point, ":", round(prediction, 4))
# Week 11 candidate
next_X = np.array([
    1.000000,
    1.000000,
    1.000000,
    0.950000
])

tree = cKDTree(X)
nearest_distance = tree.query(next_X)[0]

assert nearest_distance > 0.025

mu, std = models["Matern 5/2"].predict(
    next_X.reshape(1, -1),
    return_std=True
)

predicted_y = mu[0]**2 + std[0]**2

predicted_std = np.sqrt(
    4 * mu[0]**2 * std[0]**2
    + 2 * std[0]**4
)

print("\nWeek 11 recommended input:")
print(np.round(next_X, 6))

print("Predicted output:", predicted_y)
print("Approximate predicted std:", predicted_std)
print("Nearest distance:", nearest_distance)
print("Current maximum:", y.max())

Best observed input: [1. 1. 1. 1.]
Best observed output: 8662.4825
Matern 5/2 : 3.67**2 * Matern(length_scale=2.34, nu=2.5)
RBF : 1.72**2 * RBF(length_scale=0.804)

Symmetry probe predictions:

 Matern 5/2
[0.95 1.   1.   1.  ] : 7786.4471
[1.   0.95 1.   1.  ] : 7786.4093
[1.   1.   0.95 1.  ] : 7786.3841
[1.   1.   1.   0.95] : 7744.6282

 RBF
[0.95 1.   1.   1.  ] : 7786.3411
[1.   0.95 1.   1.  ] : 7786.373
[1.   1.   0.95 1.  ] : 7786.3779
[1.   1.   1.   0.95] : 7737.2762

Week 11 recommended input:
[1.   1.   1.   0.95]
Predicted output: 7744.628221972019
Approximate predicted std: 13.083490928020618
Nearest distance: 0.03080255752368634
Current maximum: 8662.4825


### Evaluated query

**Input:** `(1.000000, 1.000000, 1.000000, 0.950000)`  
**Output:** `7786.3713496896`

In [ ]:
X_week11 = np.array([1.  , 1.  , 1.  , 0.95])
y_week11 = np.array([7786.3713496896])
X_new, y_new = add_evaluation(X, y, X_week11, y_week11)

## Week 12

In [83]:
# All observations through Week 11
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (31, 4)

# Original Week 4 observation
reference_X = np.array([
    0.567440,
    0.818244,
    0.999466,
    0.989229
])

reference_y = 2909.8953273073485

# Swap x1 and x3
next_X = reference_X.copy()
next_X[[0, 2]] = next_X[[2, 0]]

# Check distance from existing observations
distance = cKDTree(X).query(next_X)[0]

print("Week 12 input:")
print(" ".join(f"{v:.6f}" for v in next_X))

print("\nOutput expected under symmetry:", reference_y)
print("Nearest observation distance:", distance)

# After Week 12 results are available:
# symmetry_difference = y_week12[0] - reference_y
# print("Difference:", symmetry_difference)

Week 12 input:
0.999466 0.818244 0.567440 0.989229

Output expected under symmetry: 2909.8953273073485
Nearest observation distance: 0.4236787659689826


### Evaluated query

**Input:** `(0.999466, 0.818244, 0.567440, 0.989229)`  
**Output:** `2909.8953273073485`

In [ ]:
X_week12 = np.array([0.999466, 0.818244, 0.56744 , 0.989229])
y_week12 = np.array([2909.8953273073485])
X_new, y_new = add_evaluation(X, y, X_week12, y_week12)

## Week 13

In [86]:
# Load all observations through Week 12
X_all = np.asarray(X_new, dtype=float)
y_all = np.asarray(y_new, dtype=float).ravel()

if X_all.shape != (32, 4) or y_all.shape != (32,):
    raise ValueError(
        f"Expected 32 observations. "
        f"Got X={X_all.shape}, y={y_all.shape}."
    )

print("Current best:", y_all.max())
print("Best coordinates:", X_all[np.argmax(y_all)])
# Fit the symmetric structural model
def structural(params, X):
    a, b, p = params
    X = np.atleast_2d(X)
    return (
        a * np.sum(X**p, axis=1) + b
    )**2

fit = least_squares(
    lambda params:
        structural(params, X_all) - y_all,
    x0=[26.4, -12.6, 3.93],
    bounds=(
        [0, -100, 0.1],
        [100, 100, 20]
    )
)

params = fit.x

print("Fitted coefficients:", params)
# Create global candidates
sobol = qmc.Sobol(
    d=4,
    scramble=True,
    seed=42
).random_base2(m=16)

boundary = np.array(
    list(product(
        [0, 0.25, 0.5, 0.75, 1],
        repeat=4
    )),
    dtype=float
)

candidates = np.vstack([sobol, boundary])
# Use symmetry-aware distances
X_sorted = np.unique(
    np.sort(X_all, axis=1),
    axis=0
)

candidates_sorted = np.sort(candidates, axis=1)

tree = cKDTree(X_sorted)

distances = tree.query(candidates_sorted)[0]
# Impose a minimum predicted output
predictions = structural(params, candidates)

min_predicted_output = 4000

valid = predictions >= min_predicted_output

# Find the most unexplored eligible candidate
scores = np.where(
    valid,
    distances,
    -np.inf
)

idx = np.argmax(scores)

next_X = np.round(
    candidates_sorted[idx],
    6
)
# Print final recommendation
print("\nFINAL WEEK 13 INPUT:")
print(" ".join(f"{v:.6f}" for v in next_X))

print(
    "Structural prediction:",
    structural(params, next_X)[0]
)

print(
    "Nearest symmetry-aware distance:",
    distances[idx]
)
# Check the Week 12 symmetry experiment
week4_output = float(y_week4[0])
week12_output = float(y_week12[0])

print(
    "\nWeek 12 symmetry difference:",
    week12_output - week4_output
)

Current best: 8662.4825
Best coordinates: [1. 1. 1. 1.]
Fitted coefficients: [ 26.40922037 -12.56878144   3.93406095]

FINAL WEEK 13 INPUT:
0.000000 1.000000 1.000000 1.000000
Structural prediction: 4443.406238325596
Nearest symmetry-aware distance: 0.3211097318527605

Week 12 symmetry difference: 0.0


### Week 13 evaluated result

**Input:** `(0.000000, 1.000000, 1.000000, 1.000000)`  
**Output:** `4440.5225`

The query-selection calculations above use results available through Week 12. The full record is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv).